In [1]:
# =============================================
# CELL 1 — Install AI-Toolkit (run once)
# =============================================

import subprocess, os

def run(cmd, **kwargs):
    print(f"\n▶ {cmd}\n")
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash", **kwargs)

# 1. Clone the repo (skip if already exists)
if os.path.exists("ai-toolkit"):
    print("📁 ai-toolkit folder already exists, pulling latest...")
    run("git pull", cwd="ai-toolkit")
else:
    run("git clone https://github.com/ostris/ai-toolkit.git")

# 2. Install Node.js 20 via nvm
run("curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash")
run("""
export NVM_DIR="$HOME/.nvm"
source "$NVM_DIR/nvm.sh"
nvm install 20
nvm use 20
node --version
npm --version
""")

# 3. Install Python deps
run("pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q")
run("pip install -r ai-toolkit/requirements.txt -q")
run("pip install poetry-core wheel -q")

# 4. Build the UI (using nvm node)
run("""
export NVM_DIR="$HOME/.nvm"
source "$NVM_DIR/nvm.sh"
nvm use 20
cd ai-toolkit/ui
npm install
npm run build
""")

print("\n✅ Installation complete! Run Cell 2 to start.")

📁 ai-toolkit folder already exists, pulling latest...

▶ git pull


▶ curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash


▶ 
export NVM_DIR="$HOME/.nvm"
source "$NVM_DIR/nvm.sh"
nvm install 20
nvm use 20
node --version
npm --version



▶ pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q


▶ pip install -r ai-toolkit/requirements.txt -q


▶ pip install poetry-core wheel -q


▶ 
export NVM_DIR="$HOME/.nvm"
source "$NVM_DIR/nvm.sh"
nvm use 20
cd ai-toolkit/ui
npm install
npm run build



✅ Installation complete! Run Cell 2 to start.


In [ ]:
# =============================================
# CELL 2 — Start AI-Toolkit with public URL
# =============================================

import subprocess, threading, time, requests, re, os

# Install cloudflared (direct versioned URL)
subprocess.run(
    "wget -q 'https://github.com/cloudflare/cloudflared/releases/download/2025.1.0/cloudflared-linux-amd64' -O cloudflared && chmod +x cloudflared && ./cloudflared --version",
    shell=True, check=True
)
print("✅ cloudflared ready")

PORT = 8675

# Find nvm node path
nvm_node = os.path.expanduser("~/.nvm/versions/node")
node_dirs = sorted(os.listdir(nvm_node)) if os.path.exists(nvm_node) else []
if not node_dirs:
    raise RuntimeError("nvm node not found! Re-run Cell 1.")
latest = node_dirs[-1]
node_bin = f"{nvm_node}/{latest}/bin"
print(f"✅ Using Node: {latest} from {node_bin}")

env = os.environ.copy()
env["PATH"] = f"{node_bin}:{env['PATH']}"

# Start the AI-Toolkit server
server = subprocess.Popen(
    "cd ai-toolkit/ui && npm run build_and_start",
    shell=True, executable="/bin/bash",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env
)

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Capture public URL
url_found = None
def read_tunnel():
    global url_found
    for line in iter(tunnel.stdout.readline, b""):
        decoded = line.decode("utf-8", errors="ignore")
        match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", decoded)
        if match:
            url_found = match.group(0)
            print(f"\n🌐 Your public URL: {url_found}\n")
            break

t = threading.Thread(target=read_tunnel, daemon=True)
t.start()

# Wait for server to be up
print("⏳ Waiting for server to start...")
for _ in range(90):
    try:
        requests.get(f"http://localhost:{PORT}", timeout=2)
        print(f"✅ Server is up! Open the URL above.")
        break
    except:
        time.sleep(3)

t.join(timeout=30)
if not url_found:
    print("⚠️ Could not detect URL — scroll up to find it manually.")

# Stream server logs
for line in iter(server.stdout.readline, b""):
    print(line.decode("utf-8", errors="ignore"), end="")

✅ cloudflared ready
✅ Using Node: v20.20.2 from /root/.nvm/versions/node/v20.20.2/bin
⏳ Waiting for server to start...

🌐 Your public URL: https://charming-solo-solely-sunday.trycloudflare.com

✅ Server is up! Open the URL above.

> ai-toolkit-ui@0.1.0 build_and_start
> npm install && npm run update_db && npm run build && npm run start


up to date, audited 523 packages in 2s

69 packages are looking for funding
  run `npm fund` for details

16 vulnerabilities (3 low, 3 moderate, 10 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.

> ai-toolkit-ui@0.1.0 update_db
> npx prisma generate && npx prisma db push

Prisma schema loaded from prisma/schema.prisma

✔ Generated Prisma Client (v6.3.1) to ./node_modules/@prisma/client in 79ms

Start by importing your Prisma Client (See: https://pris.ly/d/importing-client)

Tip: Curious about the SQL queries Pris